In [ ]:
import pandas as pd
import os
import warnings

# Silenciar avisos de performance do pandas
warnings.filterwarnings('ignore')

def calcular_premio_risco_v2(diretorio_dados):
    print("=== INÍCIO DO CÁLCULO DE PRÊMIO DE RISCO (FONTE EXTERNA DE CDI) ===")
    
    # 1. Atualizamos o nome do arquivo para apontar para a sua nova base Parquet
    caminho_mestre = os.path.join(diretorio_dados, "base_mestre_consolidada_IBOV_CDI_SELIC.parquet")
    
    # 2. Em vez de read_csv, usamos o comando read_parquet do Pandas para máxima velocidade
    df_mestre = pd.read_parquet(caminho_mestre)
    
    # 3. Como o arquivo Parquet anterior foi salvo de forma plana (sem índice), 
    # precisamos garantir que a coluna 'Data' seja lida como formato de calendário 
    # e configurada novamente como o índice (as "etiquetas" das linhas) da tabela
    df_mestre['Data'] = pd.to_datetime(df_mestre['Data'])
    df_mestre.set_index('Data', inplace=True)
    
    # 2. Carregamento do CDI (Arquivo isolado)
    caminho_cdi = os.path.join(diretorio_dados, "CDI_2010_2026.xlsx")
    print("2. A carregar e alinhar a taxa livre de risco (CDI)...")
    df_cdi = pd.read_excel(caminho_cdi)
    df_cdi = df_cdi.rename(columns={'Date': 'Data', 'valor': 'CDI'}).set_index('Data')
    df_cdi.index = pd.to_datetime(df_cdi.index)
    
    # 3. Cálculo dos Retornos Diários dos Ativos
    print("3. A calcular os retornos diários...")
    # Assumindo que a base mestre contém preços
    df_retornos = df_mestre.pct_change().dropna()
    
    # 4. Alinhamento e Subtração do Prêmio de Risco
    # Usamos .ffill() moderno para preencher feriados ou finais de semana vazios
    serie_cdi = df_cdi['CDI'].reindex(df_retornos.index).ffill()
    
    print("4. A subtrair o CDI (Taxa Livre de Risco) para gerar o Excess Return...")
    # Subtraímos o CDI de todas as colunas (Ações e IBOV)
    df_premio_risco = df_retornos.sub(serie_cdi, axis=0)
    
    # Renomear a coluna do IBOV para ficar claro que é o prêmio de mercado
    if 'IBOV' in df_premio_risco.columns:
        df_premio_risco = df_premio_risco.rename(columns={'IBOV': 'Premio_Mercado_IBOV'})
    
    # 5. Exportação
    caminho_saida = os.path.join(diretorio_dados, "matriz_premio_risco.csv")
    df_premio_risco.to_csv(caminho_saida)
    
    print("\n=== RESUMO DO PRÊMIO DE RISCO V2 ===")
    print(f"Período Processado: {df_premio_risco.index.min().date()} a {df_premio_risco.index.max().date()}")
    print(f"Total de Ativos: {len(df_premio_risco.columns) - 1}")
    print(f"Média do CDI diário (Benchmark): {serie_cdi.mean():.8f}")
    print(f"Arquivo gerado: {caminho_saida}")
    print("==================================================================")
    
    return df_premio_risco

# ==========================================
# EXECUÇÃO
# ==========================================
pasta_dados = r"C:\VSCodeWorkspace\TCC_Escrito\data"
base_premio = calcular_premio_risco_v2(pasta_dados)

=== INÍCIO DO CÁLCULO DE PRÊMIO DE RISCO (FONTE EXTERNA DE CDI) ===
1. A carregar ativos e IBOV da base mestre...
2. A carregar e alinhar a taxa livre de risco (CDI)...
3. A calcular os retornos diários...
4. A subtrair o CDI (Taxa Livre de Risco) para gerar o Excess Return...

=== RESUMO DO PRÊMIO DE RISCO V2 ===
Período Processado: 2010-01-05 a 2025-12-30
Total de Ativos: 136
Média do CDI diário (Benchmark): 0.00036885
Arquivo gerado: C:\VSCodeWorkspace\TCC_Escrito\data\matriz_premio_risco.csv
